# Investment Research Agent: sample queries for the team

A guide to **try the system yourself**: run the whole thing, run each piece alone, and see the behaviors worth knowing about. Run cells top to bottom the first time.

| Part | What it is | Needs API keys? |
|---|---|---|
| 0 | Setup | yes (OpenRouter, Finnhub in `.env`) |
| 1 | The overall system, end to end | yes |
| 2 | Each module on its own | mostly yes; the last three cells are offline |
| 3 | Must-see behaviors (failures, safety checks, the evaluator loop) | offline unless noted |

**Cost note:** Part 1 with the evaluator makes roughly 10 to 45 model calls per ticker (the news step is the biggest share). Finnhub free tier is 60 calls/minute, so avoid re-running Part 1 in a tight loop.

AI disclosure: this notebook was drafted with Claude (Anthropic) and reviewed by the team.

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

# Run from the repo root so `.env` and `data/` are found.
root = Path.cwd()
while root != root.parent and not (root / "pyproject.toml").exists() and not (root / "src").exists():
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root / "src"))
print("repo root:", root)

from finagent import config
config.load_environment()
for name in (config.OPENROUTER_KEY_NAME, config.FINNHUB_KEY_NAME):
    print(f"{name}:", "set" if os.environ.get(name) else "MISSING (add it to .env)")

## 1. The overall system

`run_pipeline` is the single function that chains every stage:

`profile -> plan -> tool loop -> reflection -> [memory read] -> synthesis -> [evaluator-optimizer] -> [memory write]`

### 1a. A healthy mega-cap, with the evaluator on
Things to look for: the **Plan** (why these tools), the **Trace** (which tools ran), the **Scorecard** (computed by Python, not the model), the **Evaluation** block (scores per criterion, revisions), and whether **Risks** are tied to real data.

In [ ]:
from finagent.pipeline import run_pipeline
from finagent.run_agent import format_report, format_analysis
from finagent.workflows.evaluator_optimizer import evaluate_run

run = run_pipeline("NVDA", analyze=True, evaluator=evaluate_run)  # memory off here; see 2.9
print(format_report(run.loop))
print(format_analysis(run.outcome))

In [ ]:
# The evaluator's work, round by round.
ev = run.evaluation
print("passed:", ev.passed, "| revisions:", ev.revisions, "|", ev.stopped_because)
for r in ev.rounds:
    print(f"\n{r.label}: verdict={r.verdict} overall={r.overall:.2f}")
    for s in r.judgment.scores:
        print(f"  {s['criterion']:<24} {s['score']}  {s['comment']}")
    for c in r.judgment.critical_issues:
        print("  CRITICAL:", c)

### 1b. Same system, different situations
Each ticker tests a different path. Run them one at a time.

| Ticker | Why it is interesting |
|---|---|
| `AADX` | small-cap, **loses money**: P/E is not meaningful, so signals and gaps differ |
| `ZZZZZZ` | **not a real ticker**: a fatal error that should stop cleanly, no crash |
| `SPY` | **an ETF, not a stock**: the pre-flight check should refuse it |

In [ ]:
run_aadx = run_pipeline("AADX", analyze=True, evaluator=evaluate_run)
print(format_analysis(run_aadx.outcome))
print("\nstage errors:", run_aadx.stage_errors)

In [ ]:
for bad in ("ZZZZZZ", "SPY"):
    r = run_pipeline(bad, analyze=True)
    print(bad, "->", r.loop["stop_reason"], "|", r.loop["fatal_error"])

### 1c. The same thing from a terminal
```bash
python -m finagent.run_agent NVDA                                 # data loop only
python -m finagent.run_agent NVDA --analyze                       # + the analysis
python -m finagent.run_agent NVDA --analyze --evaluate            # + judge and revision
python -m finagent.run_agent NVDA --analyze --evaluate --json     # everything as JSON
python -m finagent.run_agent NVDA --analyze --no-memory           # skip the run diary
```

## 2. Each module on its own

Run one stage at a time to see exactly what it produces. This is the best way to debug.

### 2.1 Company profile (the pre-flight check)
Decides if the ticker is a real single stock, and sets context (size, profitability, days to earnings) that shapes the plan.

In [ ]:
from finagent.tools.company_profile import get_company_profile
get_company_profile("NVDA")

### 2.2 Fundamentals (valuation, margins, growth, balance sheet)
Look at `gaps`: they say *why* a value is missing (for example `negative_earnings` for a loss-maker).

In [ ]:
from finagent.tools.fundamentals import get_fundamentals
f = get_fundamentals("AADX")
print("gaps:", f["gaps"])
f["valuation"], f["profitability"]

### 2.3 Earnings history (beat or miss vs analyst estimates)

In [ ]:
from finagent.tools.earnings_history import get_earnings_history
h = get_earnings_history("NVDA")
print(h["summary"])
for q in h["quarters"]:
    print(q)

### 2.4 Planner + research loop
The **model decides** which tools to call, in what order. `trace` shows every step; step 0 is the pre-flight check.

In [ ]:
from finagent.agent.loop import run_research_loop
loop = run_research_loop("NVDA")
print("plan emphasis:", loop["plan"]["priority_note"])
for step in loop["plan"]["steps"]:
    print(" -", step["tool"], "|", step["reason"][:90])
print()
for t in loop["trace"]:
    print(t["step"], t["tool"], "ok" if t["ok"] else "ERROR " + str(t["error"]))

### 2.5 Reflection
**Code decides** here, not the model: it retries a failed tool once if a retry can help, and reports what remains.

In [ ]:
from finagent.agent.reflection import reflect
loop2, reflection = reflect(loop)
reflection["summary"], reflection["retried"]

### 2.6 Signals (computed by Python)
Each is one checkable judgment with its source field. The model must account for every one.

In [ ]:
from finagent.agent.signals import compute_signals
for s in compute_signals(loop2["results"]):
    print(f"{s['name']:<24} {s['direction']:<9} {s['value']!s:<14} {s['why'][:70]}")

### 2.7 The news pipeline, station by station
`Ingest -> Preprocess -> Classify -> Route -> Extract -> Summarize`. Run the stations separately to see what each one removes.

In [ ]:
from finagent.workflows.news_ingest import collect_articles
batch = collect_articles("NVDA", company_name="NVIDIA")
print("kept:", len(batch.articles), "| window days:", batch.window_days, "| widened:", batch.widened)
for a in batch.articles[:5]:
    print(a.published.date(), "|", a.headline[:90])

In [ ]:
from finagent.agent.loop import make_client
from finagent.workflows.news_classify import classify_articles, make_llm_classifier

client = make_client()
classified = classify_articles(batch.articles, "NVIDIA", make_llm_classifier(client))
print("kept:", len(classified.items), "| irrelevant:", classified.irrelevant,
      "| failed:", classified.failed, "| unchecked:", classified.unchecked)
for article, label in classified.items:
    print(f"{label:<9} {article.headline[:80]}")

In [ ]:
from finagent.workflows.news_specialists import build_insights, make_llm_analyzer

insights = build_insights(classified, "NVIDIA", make_llm_analyzer(client))
for i in insights.insights:
    print(i["published"], i["category"], i["sentiment"], "|", i["summary"][:100])

In [ ]:
# The same chain as one tool call (what the agent actually uses):
from finagent.tools.news import get_news_insights
len(get_news_insights("AADX"))

### 2.8 Synthesis (the thesis)
The model writes the thesis. **Python** builds the gaps and sources, checks every evidence number, and caps the confidence.

In [ ]:
from finagent.agent.synthesis import analyze_run
outcome = analyze_run(loop2, reflection=reflection)
print(outcome.analysis["stance"], outcome.analysis["confidence"])
print(outcome.analysis["thesis"])
print("notes from Python:", outcome.notes)
print("fell back to rule-based?", outcome.fallback_reason)

### 2.9 Memory (offline, uses a temp file)
Append-only diary of past analyses; `diff_runs` says what changed since the last run. Entries older than 30 days are ignored when reading but never deleted.

In [ ]:
import tempfile
from pathlib import Path
from datetime import UTC, datetime, timedelta
from finagent.memory import record_from_outcome, save_run, load_prior, diff_runs, format_diff

tmp = Path(tempfile.mkdtemp()) / "memory.jsonl"
week_ago = datetime.now(UTC) - timedelta(days=7)

old = record_from_outcome(outcome, now=week_ago)
old["stance"] = "neutral"                      # pretend last week's view was different
save_run(old, tmp)

new = record_from_outcome(outcome)
prior = load_prior(new["ticker"], tmp)
print("\n".join(format_diff(diff_runs(prior, new))))

## 3. Must-see behaviors (mostly offline)

### 3.1 The tests
Every module has offline tests (no model, no network).

In [ ]:
!PYTHONPATH=src python -m pytest -q

### 3.2 Fail loudly, not quietly
A failed data fetch **raises**, so the trace shows `ok=False` and reflection can retry. Returning an empty result would look like success.

In [ ]:
from types import SimpleNamespace
from finagent.tools.earnings_history import get_earnings_history

class BrokenClient:
    def company_earnings(self, *a, **k):
        raise ConnectionError("network down")

try:
    get_earnings_history("NVDA", client=BrokenClient())
except RuntimeError as e:
    print("raised:", e)

### 3.3 Safety check: the model cannot invent numbers in a news summary
Every number in a summary must appear in the article, or the item is rejected.

In [ ]:
from datetime import UTC, datetime
from finagent.workflows.news_ingest import Article
from finagent.workflows.news_specialists import check_numbers_grounded, SpecialistError

article = Article(headline="Acme raises guidance", summary="Revenue guidance now 5 billion dollars.",
                  url="https://example.com", source="demo", published=datetime.now(UTC))
check_numbers_grounded("Guidance is now 5 billion dollars.", article)   # passes
try:
    check_numbers_grounded("Guidance is now 7 billion dollars.", article)  # 7 is invented
except SpecialistError as e:
    print("rejected:", e)

### 3.4 Stale news is reported as a data gap

In [ ]:
from datetime import date, timedelta
from finagent.agent.synthesis import build_data_gaps
from finagent.samples import SAMPLE_LOOP_RESULT

old_news = [{"published": (date.today() - timedelta(days=40)).isoformat(),
             "category": "general", "summary": "x", "sentiment": "neutral", "source_url": "u"}]
stale = {**SAMPLE_LOOP_RESULT, "results": {**SAMPLE_LOOP_RESULT["results"], "get_news_insights": old_news}}
[g for g in build_data_gaps(stale) if "news" in g]

### 3.5 The evaluator-optimizer loop, with a scripted judge (no API)
The **judge scores, Python decides.** A scripted judge lets you see the loop's behavior exactly: a failing draft is revised, the loop stops after at most 2 revisions, and the best draft is kept.

In [ ]:
from finagent.agent.synthesis import SynthesisOutcome
from finagent.samples import SAMPLE_ANALYSIS
from finagent.workflows.evaluation_rules import RUBRIC_CRITERIA, decide_verdict
from finagent.workflows.evaluator_optimizer import Judgment, evaluate_and_optimize

def judgment(value, critical=None):
    scores = [{"criterion": c, "score": value, "comment": f"{c} needs work"} for c in RUBRIC_CRITERIA]
    return Judgment(scores, critical_issues=critical or [])

draft = SynthesisOutcome({**SAMPLE_ANALYSIS, "stance": "neutral"}, None, [], [])
better = SynthesisOutcome({**SAMPLE_ANALYSIS, "stance": "bullish"}, None, [], [])

judgments = iter([judgment(2), judgment(5)])           # draft fails, revision passes
result = evaluate_and_optimize(draft, {}, lambda a, r, s: next(judgments), lambda fb, prev: better)
print("passed:", result.passed, "| revisions:", result.revisions, "|", result.stopped_because)
print("final stance:", result.final.analysis["stance"])

In [ ]:
# The verdict rule: mean >= 4.0, no score below 3, no critical issue.
def check(values, critical=None):
    scores = [{"criterion": c, "score": v, "comment": ""} for c, v in zip(RUBRIC_CRITERIA, values)]
    return decide_verdict(scores, critical_issues=critical)

print(check([5, 5, 5, 5, 5, 5]))                          # pass
print(check([5, 5, 5, 5, 5, 2]))                          # average is high, but one score is below 3
print(check([5, 5, 5, 5, 5, 5], ["number invented"]))     # a critical issue blocks a pass

### 3.6 Where AI helped and what the team owns
Course rule: AI contributions are disclosed in code, and decisions the team owns are marked `# AUTHOR:`.

In [ ]:
!grep -rn "AI DISCLOSURE" src | wc -l
!grep -rn "# AUTHOR" src | head -20

### 3.7 Known limits (worth stating in the write-up)
- **Same model writes and judges** by default (`config.JUDGE_MODEL`), which can make the judge lenient on its own writing.
- The judge is **not calibrated against human ratings** yet.
- The number check catches **invented numbers**, not invented non-numeric claims.
- Finnhub's free news feed is **capped and noisy** (roundups tagged with the ticker), so articles are ranked by company mentions before the cap.
- News summaries skew **positive** (press releases), so news sentiment is weak evidence and is not part of the scorecard.
- Free-tier limits: Finnhub 60 calls/min, Alpha Vantage 25/day, NewsAPI ~100/day; yfinance is unofficial.